# 예지 제어 RL 환경 노트북

**목표**: 이상 센서값을 정상 밴드로 복귀시키는 강화학습 환경을 정의하고 PPO 베이스라인을 학습합니다.

**정상 밴드**
- 온도: 27–33
- 습도: 40–50
- 진동: 2–4

**위험 한계**
- 온도: (5, 75)
- 습도: (5, 80)
- 진동: (0.5, 8)


In [ ]:
# 의존성 import
try:
    import gymnasium as gym
except ImportError:  # fallback
    import gym

import numpy as np
from gym import spaces

from stable_baselines3 import PPO


In [ ]:
# 환경 상수 정의
NORMAL_BAND = {
    "temp": (27.0, 33.0),
    "hum": (40.0, 50.0),
    "vib": (2.0, 4.0),
}

RISK_LIMITS = {
    "temp": (5.0, 75.0),
    "hum": (5.0, 80.0),
    "vib": (0.5, 8.0),
}

# 행동 수치 (README 기반) - 퍼센트 변화 범위
LEVEL_RANGES = {
    "actuator": {
        1: (0.01, 0.20),
        2: (0.21, 0.40),
        3: (0.41, 0.60),
    },
    "rail": {
        1: (-0.20, 0.30),
        2: (-0.20, -0.10),
        3: (0.0, 0.0),
        4: (0.01, 0.10),
        5: (0.11, 0.20),
        6: (0.21, 0.30),
    },
}


In [ ]:
class PredictiveControlEnv(gym.Env):
    """
    예지 제어용 Gym 환경

    분류 라벨 및 제어 행동 (README 기반):
    - 온도 급상승 -> TEMP_RUNAWAY -> 쿨링
    - 습도 상승, 온도 하락 -> HUM_UP_TEMP_DOWN -> 제습기, 히터
    - 진동 상승, 온도 상승 -> VIB_RISE_TEMP_LAG -> 레일 속도 조절, 쿨링
    - 진동 스파이크 -> VIB_SPIKE_REPEAT -> 레일 속도 조절 or stop

    Action Space
    - MultiDiscrete([4, 4, 4, 6])
      * cooling: 0~3 (0=off)
      * heater: 0~3 (0=off)
      * dehumidifier: 0~3 (0=off)
      * rail gear: 0~5 -> 실제 기어 1~6

    Observation (연속 Box, rail gear/label은 정수지만 float 캐스팅해 사용)
    - temp, hum, vib, rail gear
    - prev controls: heater, cooling, dehumidifier, rail gear
    - label
    """

    metadata = {"render_modes": []}

    def __init__(self, max_steps: int = 200):
        super().__init__()
        self.max_steps = max_steps

        low = np.array(
            [
                RISK_LIMITS["temp"][0],
                RISK_LIMITS["hum"][0],
                RISK_LIMITS["vib"][0],
                1.0,  # rail gear
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,  # label
            ],
            dtype=np.float32,
        )
        high = np.array(
            [
                RISK_LIMITS["temp"][1],
                RISK_LIMITS["hum"][1],
                RISK_LIMITS["vib"][1],
                6.0,
                3.0,
                3.0,
                3.0,
                6.0,
                3.0,  # label (0~3 예시)
            ],
            dtype=np.float32,
        )

        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)
        self.action_space = spaces.MultiDiscrete([4, 4, 4, 6])

        self.reset()

    def _round_clip(self, value: float, key: str) -> float:
        clipped = np.clip(value, *RISK_LIMITS[key])
        return float(np.round(clipped, 3))

    def _random_outside_normal(self, key: str) -> float:
        low, high = RISK_LIMITS[key]
        n_low, n_high = NORMAL_BAND[key]
        if np.random.rand() < 0.5:
            sample = np.random.uniform(low, n_low - 0.5)
        else:
            sample = np.random.uniform(n_high + 0.5, high)
        return self._round_clip(sample, key)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.temp = self._random_outside_normal("temp")
        self.hum = self._random_outside_normal("hum")
        self.vib = self._random_outside_normal("vib")
        self.rail_gear = np.random.randint(1, 7)

        self.prev_heater = 0
        self.prev_cooling = 0
        self.prev_dehumidifier = 0
        self.prev_rail = self.rail_gear
        self.label = np.random.randint(0, 4)

        self.steps = 0
        self.consecutive_in_band = 0
        self.steps_since_in_band = 0

        return self._get_obs(), {}

    def _apply_actuator(self, value: float, level: int, key: str, direction: int) -> float:
        if level == 0:
            return value
        min_p, max_p = LEVEL_RANGES["actuator"][level]
        pct = np.random.uniform(min_p, max_p)
        return value + direction * value * pct

    def _apply_rail(self, vib: float, gear: int) -> float:
        min_p, max_p = LEVEL_RANGES["rail"][gear]
        pct = np.random.uniform(min_p, max_p)
        return vib + vib * pct

    def _in_normal_band(self) -> bool:
        return (
            NORMAL_BAND["temp"][0] <= self.temp <= NORMAL_BAND["temp"][1]
            and NORMAL_BAND["hum"][0] <= self.hum <= NORMAL_BAND["hum"][1]
            and NORMAL_BAND["vib"][0] <= self.vib <= NORMAL_BAND["vib"][1]
        )

    def step(self, action):
        cooling, heater, dehumidifier, rail_action = action
        rail_gear = int(rail_action) + 1

        prev_in_band = self._in_normal_band()

        self.temp = self._apply_actuator(self.temp, int(cooling), "temp", direction=-1)
        self.temp = self._apply_actuator(self.temp, int(heater), "temp", direction=1)
        self.hum = self._apply_actuator(self.hum, int(dehumidifier), "hum", direction=-1)
        self.vib = self._apply_rail(self.vib, rail_gear)

        self.temp = self._round_clip(self.temp, "temp")
        self.hum = self._round_clip(self.hum, "hum")
        self.vib = self._round_clip(self.vib, "vib")

        self.prev_heater = int(heater)
        self.prev_cooling = int(cooling)
        self.prev_dehumidifier = int(dehumidifier)
        self.prev_rail = rail_gear
        self.rail_gear = rail_gear

        self.steps += 1

        in_band = self._in_normal_band()
        if in_band:
            self.consecutive_in_band += 1
            self.steps_since_in_band = 0
        else:
            self.consecutive_in_band = 0
            self.steps_since_in_band += 1

        reward = self._compute_reward(prev_in_band, in_band)

        terminated, info = self._check_termination(in_band)
        if terminated:
            if info.get("termination") == "success":
                reward += 500
            elif info.get("termination") == "failure":
                reward -= 600
        truncated = self.steps >= self.max_steps

        return self._get_obs(), reward, terminated, truncated, info

    def _distance_to_band(self, value: float, band: tuple) -> float:
        if band[0] <= value <= band[1]:
            return 0.0
        if value < band[0]:
            return band[0] - value
        return value - band[1]

    def _compute_reward(self, prev_in_band: bool, in_band: bool) -> float:
        dist = (
            self._distance_to_band(self.temp, NORMAL_BAND["temp"])
            + self._distance_to_band(self.hum, NORMAL_BAND["hum"])
            + self._distance_to_band(self.vib, NORMAL_BAND["vib"])
        )
        reward = -dist

        if not prev_in_band and in_band:
            reward += 200
        if prev_in_band and not in_band:
            reward -= 200

        if self._exceeds_risk():
            reward -= 300

        return reward

    def _exceeds_risk(self) -> bool:
        return (
            not (RISK_LIMITS["temp"][0] <= self.temp <= RISK_LIMITS["temp"][1])
            or not (RISK_LIMITS["hum"][0] <= self.hum <= RISK_LIMITS["hum"][1])
            or not (RISK_LIMITS["vib"][0] <= self.vib <= RISK_LIMITS["vib"][1])
        )

    def _check_termination(self, in_band: bool):
        info = {}
        if self._exceeds_risk():
            info["termination"] = "risk_limit"
            return True, info

        if self.consecutive_in_band >= 30:
            info["termination"] = "success"
            return True, info

        if self.steps_since_in_band >= 30:
            info["termination"] = "failure"
            return True, info

        return False, info

    def _get_obs(self):
        obs = np.array(
            [
                self.temp,
                self.hum,
                self.vib,
                float(self.rail_gear),
                float(self.prev_heater),
                float(self.prev_cooling),
                float(self.prev_dehumidifier),
                float(self.prev_rail),
                float(self.label),
            ],
            dtype=np.float32,
        )
        return obs


In [ ]:
# 보상 설계는 step 내 _compute_reward에서 적용됨
# - 정상 밴드 가까워질수록 보상 증가 (거리 기반)
# - 위험 한계 초과 패널티
# - 정상 밴드 외부->내부 +200, 내부->외부 -200
# - 성공 종료 +500, 실패 종료 -600 (step 내 종료 처리에서 적용)


In [ ]:
# 종료 조건 구현 확인

env = PredictiveControlEnv(max_steps=200)


def run_rollout(env, steps=10):
    obs, _ = env.reset()
    total_reward = 0.0
    for _ in range(steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    return total_reward

run_rollout(env, steps=20)


In [ ]:
# PPO baseline 학습 코드

env = PredictiveControlEnv(max_steps=200)

model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=2000)

# 간단 rollout 시연
obs, _ = env.reset()
for _ in range(5):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        break
